# CIFAR-10 Example
This notebook demonstrates VGG11 trained on CIFAR-10 with the **Theta** optimizer — a minimal wrapper that injects truncated [Lomax (Type-II Pareto)](https://en.wikipedia.org/wiki/Lomax_distribution) heavy-tailed stochastic gradient noise (SGN) into any PyTorch optimizer, proposed in [WOR22](https://arxiv.org/abs/2102.04297) and [WR25](https://arxiv.org/abs/2510.20905).

### TL; DR
> Injection of multiplicative and anisotropic heavy-tailed SGN with subsequent truncation is effective for SGD to escape from sharp minima.

### Curation of Heavy-tailed Noise
$$
g_{heavy} = g_{SB_a} + Z(g_{SB} - g_{LB}) \; (\approx g_{SB_a} + Z(g_{SB} - \nabla F(\theta)))
$$
- Method 1: $\theta_{t+1} = \theta_t - \eta\cdot \varphi_b( g_{heavy}(\theta_t))$, where $g_{SB} \ne g_{SB_a}$
- Method 2: $\theta_{t+1} = \theta_t - \eta\cdot \varphi_b(g_{heavy}(\theta_t))$, where $g_{SB} = g_{SB_a}$
- SB + Clip: $\theta_{t+1} = \theta_t - \eta\cdot \varphi_b(g_{SB}(\theta_t))$
- SB + Noise: $\theta_{t+1} = \theta_t - \eta\cdot  g_{heavy}(\theta_t)$, where $g_{SB} = g_{SB_a}$
- LB: $\theta_{t+1} = \theta_t - \eta\cdot g_{LB}(\theta_t)$
- SB: $\theta_{t+1} = \theta_t - \eta\cdot g_{SB}(\theta_t)$
- ($\varphi_b(\cdot)$: gradient norm clipping operator with clip threshold equals to $b$; $\eta$: step size)

In [ ]:
import time
from itertools import cycle

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

from tqdm.auto import tqdm

from theta import Theta, NoiseStep

In [ ]:
torch.manual_seed(5959)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Keep the demo quick. Remove the Subset wrappers for full CIFAR-10 runs.
#train_dataset = Subset(train_dataset, range(512))
#test_dataset = Subset(test_dataset, range(512))

train_loader_sb = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
train_loader_lb = DataLoader(train_dataset, batch_size=256, shuffle=True, drop_last=True)
train_loader_sb_a = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
train_loader_sb_b = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)

train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)  # sharpness eval
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"train={len(train_dataset)}, test={len(test_dataset)}")

In [ ]:
def build_vgg11(num_classes=10):
    model = models.vgg11(weights=None)
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    model.classifier = nn.Linear(512, num_classes)
    return model

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
def batch_to_device(batch, device):
    x, y = batch
    return x.to(device), y.to(device)


def loss_on_batch(model, batch, criterion, device):
    x, y = batch_to_device(batch, device)
    logits = model(x)
    loss = criterion(logits, y)
    acc = logits.argmax(dim=1).eq(y).float().mean().item() * 100.0
    return loss, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        x, y = batch_to_device(batch, device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += logits.argmax(dim=1).eq(y).sum().item()
        total += y.size(0)
    model.train()
    return total_loss / total, 100.0 * total_correct / total


def expected_sharpness(model, loader, criterion, device, delta=0.01, num_samples=5):
    model.eval()
    base_loss, _ = evaluate(model, loader, criterion, device)
    params = [p for p in model.parameters() if p.requires_grad]
    total_diff = 0.0

    for _ in range(num_samples):
        noises = []
        with torch.no_grad():
            for p in params:
                noise = torch.randn_like(p).mul_(delta)
                p.add_(noise)
                noises.append(noise)

        perturbed_loss, _ = evaluate(model, loader, criterion, device)
        total_diff += abs(perturbed_loss - base_loss)

        with torch.no_grad():
            for p, noise in zip(params, noises):
                p.sub_(noise)

    model.train()
    return total_diff / num_samples

## Replication Study

In [ ]:
MAX_STEPS = 30_000
NOISE_STOP_STEP = 25_000
LEARNING_RATE = 0.01
CLIP_THRESHOLD = 0.25
TAIL_PROBABILITY = 0.9

#### 1) SB

In [ ]:
sb_model = build_vgg11().to(device)
sb_optimizer = optim.SGD(sb_model.parameters(), lr=LEARNING_RATE)

data_iter = cycle(train_loader_sb)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="SB")
for step in pbar:
    loss, acc = loss_on_batch(sb_model, next(data_iter), criterion, device)
    sb_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    sb_optimizer.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.1f}")

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(sb_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(sb_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### 2) LB

In [ ]:
lb_model = build_vgg11().to(device)
lb_optimizer = optim.SGD(lb_model.parameters(), lr=LEARNING_RATE)

data_iter = cycle(train_loader_lb)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="LB")
for step in pbar:
    loss, acc = loss_on_batch(lb_model, next(data_iter), criterion, device)
    lb_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    lb_optimizer.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.1f}")

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(lb_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(lb_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### 3) SB + Clip

In [ ]:
sb_clip_model = build_vgg11().to(device)
sb_clip_optimizer = Theta(
    sb_clip_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

data_iter = cycle(train_loader_sb)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="SB + Clip")
for step in pbar:
    loss, acc = loss_on_batch(sb_clip_model, next(data_iter), criterion, device)
    sb_clip_optimizer.zero_grad()
    loss.backward()
    sb_clip_optimizer.step()  # clips then steps
    pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.1f}")

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(sb_clip_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(sb_clip_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### 4) SB + Noise

In [ ]:
sb_noise_model = build_vgg11().to(device)
sb_noise_optimizer = Theta(
    sb_noise_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=None,
    tail_prob=0.0,
)

current_iter = cycle(train_loader_sb_a)
plus_iter = cycle(train_loader_sb)
minus_iter = cycle(train_loader_lb)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="SB + Noise | p=0.0")
for step in pbar:
    noise = sb_noise_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(sb_noise_model, next(current_iter), criterion, device)

    sb_noise_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(sb_noise_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB
        loss_plus, _ = loss_on_batch(sb_noise_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    sb_noise_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(sb_noise_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(sb_noise_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### 5) Method 1
$$
g_{heavy} = g_{SB_a} + Z(g_{SB} - g_{LB})
$$

In [ ]:
method_1_model = build_vgg11().to(device)
method_1_optimizer = Theta(
    method_1_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB (independent from g_SB_a)
minus_iter = cycle(train_loader_lb)      # g_LB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="Method 1 | p=0.0")
for step in pbar:
    noise = method_1_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(method_1_model, next(current_iter), criterion, device)

    method_1_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(method_1_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB
        loss_plus, _ = loss_on_batch(method_1_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    method_1_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(method_1_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(method_1_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### 6) Method 2
$$
g_{heavy} = g_{SB} + Z(g_{SB} - g_{LB})
$$

In [ ]:
method_2_model = build_vgg11().to(device)
method_2_optimizer = Theta(
    method_2_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

current_iter = cycle(train_loader_sb)    # g_SB = g_SB_a (reused)
minus_iter = cycle(train_loader_lb)      # g_LB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="Method 2 | p=0.0")
for step in pbar:
    noise = method_2_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(method_2_model, next(current_iter), criterion, device)

    method_2_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(method_2_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB (reuse current batch)
        (z * loss).backward(retain_graph=True)

    # anchor gradient: g_SB
    loss.backward()
    method_2_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(method_2_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(method_2_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

## Variants for Efficiency
### A) Cheap approximation of SGN without LB
- A faithful computation of SGN requires using SB and LB, on top of SB_a, i.e., three different batches and three backward passes.
- To avoid the prohibitive computation, Method 2 already proposed to reuse as SB = SB_a, but similarly it can be further saved.
- For example, replacing LB with other SB; or replacing SGN with other SB.

### B) Noise activation using threshold
- As the SGN computation is expensive, a trick of threshold-based noise injection (proposed in section 3.2 of [WR25](https://arxiv.org/abs/2510.20905)) is implemented using `tail_prob`.
- It is a Lomax CDF threshold: after sampling $Z$, noise is injected only when: 
$$ 
\operatorname{CDF}(Z) \ge \mathtt{tail\_prob} \\
( \text{i.e., } Z\cdot\mathbf{I}\{ Z > \operatorname{CDF}^{-1}( \mathtt{tail\_prob} ) \} )
$$   
- Thus, inactive steps skip the extra stochastic-gradient noise computations, and reduces to vanilla SGD.

#### A-1) SB instead of LB
$$
g_{heavy} = g_{SB_a} + Z(g_{SB} - g_{SB_b})
$$

In [ ]:
a1_model = build_vgg11().to(device)
a1_optimizer = Theta(
    a1_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB
minus_iter = cycle(train_loader_sb_b)    # g_SB_b (replaces g_LB)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="A-1: SB instead of LB | p=0.0")
for step in pbar:
    noise = a1_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(a1_model, next(current_iter), criterion, device)

    a1_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_SB_b
        loss_minus, _ = loss_on_batch(a1_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB
        loss_plus, _ = loss_on_batch(a1_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    a1_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(a1_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(a1_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### A-2) SB instead of SGN
$$
g_{heavy} = g_{SB_a} + Zg_{SB}
$$

In [ ]:
a2_model = build_vgg11().to(device)
a2_optimizer = Theta(
    a2_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="A-2: SB instead of SGN | p=0.0")
for step in pbar:
    noise = a2_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(a2_model, next(current_iter), criterion, device)

    a2_optimizer.zero_grad()

    if noise.active:
        # plus direction only: +Z * g_SB (no minus term)
        loss_plus, _ = loss_on_batch(a2_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    a2_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(a2_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(a2_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### B-1) Method 1 with thresholding
$$
g_{heavy} = g_{SB_a} + Z\cdot\mathbf{I}\{ Z > C \} (g_{SB} - g_{LB})
$$

In [ ]:
b1_model = build_vgg11().to(device)
b1_optimizer = Theta(
    b1_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB
minus_iter = cycle(train_loader_lb)      # g_LB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"B-1: Method 1 with thresholding | p={TAIL_PROBABILITY}")
for step in pbar:
    noise = b1_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(b1_model, next(current_iter), criterion, device)

    b1_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(b1_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB
        loss_plus, _ = loss_on_batch(b1_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    b1_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(b1_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(b1_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### B-2) Method 2 with thresholding
$$
g_{heavy} = g_{SB} + Z\cdot\mathbf{I}\{ Z > C \} (g_{SB} - g_{LB})
$$

In [ ]:
b2_model = build_vgg11().to(device)
b2_optimizer = Theta(
    b2_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

current_iter = cycle(train_loader_sb)    # g_SB = g_SB_a (reused)
minus_iter = cycle(train_loader_lb)      # g_LB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"B-2: Method 2 with thresholding | p={TAIL_PROBABILITY}")
for step in pbar:
    noise = b2_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(b2_model, next(current_iter), criterion, device)

    b2_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(b2_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB (reuse current batch)
        (z * loss).backward(retain_graph=True)

    # anchor gradient: g_SB
    loss.backward()
    b2_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(b2_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(b2_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### B-3) Method A-1 with thresholding
$$
g_{heavy} = g_{SB_a} + Z\cdot\mathbf{I}\{ Z > C \} (g_{SB} - g_{SB_b})
$$

In [ ]:
b3_model = build_vgg11().to(device)
b3_optimizer = Theta(
    b3_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB
minus_iter = cycle(train_loader_sb_b)    # g_SB_b (replaces g_LB)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"B-3: Method A-1 with thresholding | p={TAIL_PROBABILITY}")
for step in pbar:
    noise = b3_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(b3_model, next(current_iter), criterion, device)

    b3_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_SB_b
        loss_minus, _ = loss_on_batch(b3_model, next(minus_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB
        loss_plus, _ = loss_on_batch(b3_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    b3_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(b3_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(b3_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### B-4) Method A-2 with thresholding
$$
g_{heavy} = g_{SB_a} + Z\cdot\mathbf{I}\{ Z > C \} g_{SB}
$$

In [ ]:
b4_model = build_vgg11().to(device)
b4_optimizer = Theta(
    b4_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

current_iter = cycle(train_loader_sb_a)  # g_SB_a
plus_iter = cycle(train_loader_sb)       # g_SB
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"B-4: Method A-2 with thresholding | p={TAIL_PROBABILITY}")
for step in pbar:
    noise = b4_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value

    loss, acc = loss_on_batch(b4_model, next(current_iter), criterion, device)

    b4_optimizer.zero_grad()

    if noise.active:
        # plus direction only: +Z * g_SB (no minus term)
        loss_plus, _ = loss_on_batch(b4_model, next(plus_iter), criterion, device)
        (z * loss_plus).backward()

    # anchor gradient: g_SB_a
    loss.backward()
    b4_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(b4_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(b4_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)